# TATN 预训练阶段完整复现（留一法，对照论文）
**评估方式**：9折留一法（Paper Section IV.D），每次用8条完整 drive cycle 训练，留1条测试。

**运行前**：Runtime → Change runtime type → GPU

## Step 1：挂载 Google Drive，设置路径

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_RAW_DATA = '/content/drive/MyDrive/Research/mining review/TATN/Panasonic 18650PF Data'
DRIVE_RESULTS  = '/content/drive/MyDrive/Research/mining review/TATN/results/pretrain_loo'
WORK_DIR = '/content/TATN'

print('根目录内容：')
for item in sorted(os.listdir(DRIVE_RAW_DATA)):
    print(' ', item)

Mounted at /content/drive
根目录内容：
  -10degC
  -20degC
  -20degC Trise
  -20degC Trise with pause
  .DS_Store
  0degC
  10degC
  10degC Trise
  10degC Trise with pause
  Panasonic 18650PF Data
  Plot_results.m
  Readme file - desc of tests performed.txt


## Step 2：拉取代码

In [2]:
import shutil
os.chdir('/content')
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
!git clone https://github.com/zhangcastle/TATN.git /content/TATN -q
os.chdir(WORK_DIR)
print('工作目录:', os.getcwd())

工作目录: /content/TATN


## Step 3：安装依赖

In [3]:
!pip install scipy tqdm scikit-learn -q
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: NVIDIA L4


## Step 4：探索原始数据结构

In [4]:
def find_drive_cycles_dir(base, temp_folder):
    for sub in ['Drive cycles', 'Drive Cycles', 'drive cycles', '']:
        p = os.path.join(base, temp_folder, sub) if sub else os.path.join(base, temp_folder)
        if os.path.exists(p) and any(f.endswith('.mat') for f in os.listdir(p)):
            return p
    return None

temp_raw = {
    '25':  find_drive_cycles_dir(os.path.join(DRIVE_RAW_DATA, 'Panasonic 18650PF Data'), '25degC'),
    '10':  find_drive_cycles_dir(DRIVE_RAW_DATA, '10degC'),
    '0':   find_drive_cycles_dir(DRIVE_RAW_DATA, '0degC'),
    'n10': find_drive_cycles_dir(DRIVE_RAW_DATA, '-10degC'),
    'n20': find_drive_cycles_dir(DRIVE_RAW_DATA, '-20degC'),
}

for temp, path in temp_raw.items():
    print(f'\n=== {temp} ===')
    if path:
        for f in sorted(os.listdir(path)):
            if f.endswith('.mat'): print(' ', f)
    else:
        print('  [未找到]')


=== 25 ===
  03-18-17_02.17 25degC_Cycle_1_Pan18650PF.mat
  03-19-17_03.25 25degC_Cycle_2_Pan18650PF.mat
  03-19-17_09.07 25degC_Cycle_3_Pan18650PF.mat
  03-19-17_14.31 25degC_Cycle_4_Pan18650PF.mat
  03-20-17_01.43 25degC_US06_Pan18650PF.mat
  03-20-17_05.56 25degC_HWFTa_Pan18650PF.mat
  03-20-17_19.27 25degC_HWFTb_Pan18650PF.mat
  03-21-17_00.29 25degC_UDDS_Pan18650PF.mat
  03-21-17_09.38 25degC_LA92_Pan18650PF.mat
  03-21-17_16.27 25degC_NN_Pan18650PF.mat

=== 10 ===
  03-27-17_09.06 10degC_HWFET_Pan18650PF.mat
  03-27-17_09.06 10degC_LA92_Pan18650PF.mat
  03-27-17_09.06 10degC_NN_Pan18650PF.mat
  03-27-17_09.06 10degC_UDDS_Pan18650PF.mat
  03-27-17_09.06 10degC_US06_HWFET_UDDS_LA92_NN_Pan18650PF.mat
  03-27-17_09.06 10degC_US06_Pan18650PF.mat
  03-28-17_12.51 10degC_Cycle_1_Pan18650PF.mat
  03-28-17_18.18 10degC_Cycle_2_Pan18650PF.mat
  04-05-17_17.04 10degC_Cycle_3_Pan18650PF.mat
  04-05-17_22.50 10degC_Cycle_4_Pan18650PF.mat

=== 0 ===
  05-30-17_12.56 0degC_Cycle_1_Pan18650PF.m

## Step 5：数据处理（保存完整 drive cycle，不做 train/test 时间切分）

In [5]:
import scipy.io as sio
import numpy as np

CYCLE_KW = {
    'Cycle_1': ['Cycle_1','Cycle1'], 'Cycle_2': ['Cycle_2','Cycle2'],
    'Cycle_3': ['Cycle_3','Cycle3'], 'Cycle_4': ['Cycle_4','Cycle4'],
    'NN':      ['_NN_','_NN.'],      'US06':    ['US06'],
    'HWFET':   ['HWFET','HWFTa','HWFTb'],
    'LA92':    ['LA92'],             'UDDS':    ['UDDS'],
}

def match_cycle(fname):
    matched = [c for c, kws in CYCLE_KW.items() if any(kw in fname for kw in kws)]
    return matched[0] if len(matched) == 1 else (f'SKIP' if len(matched) > 1 else None)

def read_mat_file(fpath):
    data = sio.loadmat(fpath)
    if 'meas' in data:
        m = data['meas']
        return (m['Time'][0][0].flatten(), m['Current'][0][0].flatten(),
                m['Voltage'][0][0].flatten(), m['Battery_Temp_degC'][0][0].flatten(),
                m['Ah'][0][0].flatten())
    return (data['time'].flatten(), data['current'].flatten(),
            data['voltage'].flatten(), data['temp'].flatten(), data['ah'].flatten())

def process_temp(raw_path, temp_label, out_base, interval=10):
    """保存完整 drive cycle（不做 train/test 时间切分）"""
    out_path = os.path.join(out_base, temp_label)
    os.makedirs(out_path, exist_ok=True)
    mat_files = sorted([f for f in os.listdir(raw_path) if f.endswith('.mat')])
    all_data = {}
    for fname in mat_files:
        cycle = match_cycle(fname)
        if cycle is None or cycle == 'SKIP':
            if cycle == 'SKIP': print(f'  [跳过合并] {fname}')
            else: print(f'  [未识别] {fname}')
            continue
        t, cur, vol, tmp, ah = read_mat_file(os.path.join(raw_path, fname))
        t2=t[::interval]; cur=cur[::interval]; vol=vol[::interval]
        tmp=tmp[::interval]; ah=ah[::interval]
        idx = next((i for i in range(len(t2)-1) if t2[i+1]-t2[i] < 2), 0)
        seg = (cur[idx:], vol[idx:], tmp[idx:], ah[idx:])
        if cycle in all_data:
            all_data[cycle] = tuple(np.concatenate([all_data[cycle][i], seg[i]]) for i in range(4))
            print(f'  [{temp_label}] 合并: {fname} -> {cycle}  total={len(all_data[cycle][0])}')
        else:
            all_data[cycle] = seg
            print(f'  [{temp_label}] 读取: {fname} -> {cycle}  n={len(seg[0])}')
    if not all_data: print(f'[{temp_label}] 无数据'); return
    # 全局 min/max（按温度）
    def grange(i): v=np.concatenate([d[i] for d in all_data.values()]); return v.min(),v.max()
    ranges = [grange(i) for i in range(4)]
    norm = lambda x,r: (x-r[0])/(r[1]-r[0])
    for cycle,(cur,vol,tmp,ah) in all_data.items():
        cn,vn,tn,an = norm(cur,ranges[0]),norm(vol,ranges[1]),norm(tmp,ranges[2]),norm(ah,ranges[3])
        # 保存完整文件（无切分）
        sio.savemat(os.path.join(out_path, f'{temp_label}{cycle}.mat'),
            {'current':cn.reshape(-1,1),'voltage':vn.reshape(-1,1),
             'temp':tn.reshape(-1,1),'ah':an.reshape(-1,1)})
        print(f'  saved {cycle}.mat  n={len(an)}')
    print(f'[{temp_label}] done\n')

OUTPUT_BASE = os.path.join(WORK_DIR, 'normalized_data', 'Pan')
for temp_label, raw_path in temp_raw.items():
    print(f'========== {temp_label}°C ==========')
    if raw_path: process_temp(raw_path, temp_label, OUTPUT_BASE)
    else: print('路径未找到，跳过\n')
print('全部处理完成')

========== 25°C ==========
  [25] 读取: 03-18-17_02.17 25degC_Cycle_1_Pan18650PF.mat -> Cycle_1  n=10965
  [25] 读取: 03-19-17_03.25 25degC_Cycle_2_Pan18650PF.mat -> Cycle_2  n=11127
  [25] 读取: 03-19-17_09.07 25degC_Cycle_3_Pan18650PF.mat -> Cycle_3  n=10245
  [25] 读取: 03-19-17_14.31 25degC_Cycle_4_Pan18650PF.mat -> Cycle_4  n=12087
  [25] 读取: 03-20-17_01.43 25degC_US06_Pan18650PF.mat -> US06  n=4807
  [25] 读取: 03-20-17_05.56 25degC_HWFTa_Pan18650PF.mat -> HWFET  n=7596
  [25] 合并: 03-20-17_19.27 25degC_HWFTb_Pan18650PF.mat -> HWFET  total=15178
  [25] 读取: 03-21-17_00.29 25degC_UDDS_Pan18650PF.mat -> UDDS  n=22419
  [25] 读取: 03-21-17_09.38 25degC_LA92_Pan18650PF.mat -> LA92  n=14088
  [25] 读取: 03-21-17_16.27 25degC_NN_Pan18650PF.mat -> NN  n=11699
  saved Cycle_1.mat  n=10965
  saved Cycle_2.mat  n=11127
  saved Cycle_3.mat  n=10245
  saved Cycle_4.mat  n=12087
  saved US06.mat  n=4807
  saved HWFET.mat  n=15178
  saved UDDS.mat  n=22419
  saved LA92.mat  n=14088
  saved NN.mat  n=11699
[25

## Step 6：留一法预训练（5 个温度 × 9 折 = 45 次训练）

每折完成后立刻保存模型到 Drive。预计总耗时 1-2 小时（L4 GPU）。

In [ ]:
import sys, time, glob
import torch.nn as nn, torch.optim as optim
sys.path.insert(0, WORK_DIR)
import importlib
import mydata, models
from pretrain import pretrain_loo
importlib.reload(mydata)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
EPOCHS, BATCH_SIZE, LR, EVAL_INTERVAL = 2000, 64, 0.001, 200
TEMPS = ['25', '10', '0', 'n10', 'n20']
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Device: {DEVICE}  Epochs: {EPOCHS}  Folds: 9 per temp')

In [ ]:
importlib.reload(mydata)
importlib.reload(models)
all_results = {}

for temp in TEMPS:
    print(f'\n{"#"*60}\n# 预训练温度: {temp}°C\n{"#"*60}')
    if not os.path.exists(os.path.join(OUTPUT_BASE, temp)):
        print('数据不存在，跳过'); continue
    t0 = time.time()
    result = pretrain_loo(
        rundir=f'pretrain_loo_{temp}',
        source_temp=temp,
        source_data_path=OUTPUT_BASE + '/',
        all_set=mydata.Pan_all_set,
        lr=LR, batch_size=BATCH_SIZE, epochs=EPOCHS,
        eval_interval=EVAL_INTERVAL, seed=100,
        device_type=DEVICE, ifsave=True,
    )
    elapsed = (time.time()-t0)/60
    all_results[temp] = result
    # 保存所有 fold 的模型到 Drive
    fold_pts = glob.glob(os.path.join(WORK_DIR, f'run/pretrain_loo_{temp}_fold*/saved_model/best.pt'))
    for pt in fold_pts:
        fold_name = pt.split('/')[-3]  # e.g. pretrain_loo_25_fold1_Cycle_1
        dest = os.path.join(DRIVE_RESULTS, f'{fold_name}.pt')
        shutil.copy(pt, dest)
    print(f'[{temp}°C] {len(fold_pts)} 个模型已保存到 Drive，耗时 {elapsed:.1f} min')

print('\n========== 全部完成 ==========')
print('温度  |  avg MAE  |  avg RMSE  |  论文参考')
print('-'*50)
for temp, res in all_results.items():
    if res:
        print(f"{temp:>4}°C | {res['avg_mae']*100:>7.3f}%  | {res['avg_rmse']*100:>8.3f}%   | MAE~1.09% RMSE~1.44%")

---
## Step 7：源域全量预训练（迁移阶段用）

用每个温度的**全部 9 个完整 cycle** 训练，每个温度保存 1 个模型，共 5 个。
这是迁移阶段的起点（对应论文 Section III.A）。

In [6]:
import sys, time, shutil, glob
import torch, torch.nn as nn, torch.optim as optim
sys.path.insert(0, WORK_DIR)
import importlib
import mydata, models
from pretrain import pretrain
importlib.reload(mydata)
importlib.reload(models)
importlib.reload(sys.modules['pretrain'])
from pretrain import pretrain

DRIVE_SRC_MODELS = '/content/drive/MyDrive/Research/mining review/TATN/results/source_models'
os.makedirs(DRIVE_SRC_MODELS, exist_ok=True)

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
EPOCHS, BATCH_SIZE, LR, EVAL_INTERVAL = 2000, 64, 0.001, 200
TEMPS = ['25', '10', '0', 'n10', 'n20']

for temp in TEMPS:
    print(f'\n{"="*55}\n源域预训练: {temp}°C (全部9个cycle)\n{"="*55}')
    if not os.path.exists(os.path.join(OUTPUT_BASE, temp)):
        print('数据不存在，跳过'); continue
    importlib.reload(models)
    importlib.reload(sys.modules['pretrain'])
    from pretrain import pretrain
    mdls = {
        'conv':models.conv(),'lstm':models.lstm(),'fc':models.fc(),
        'regression':models.regression(),'conv_s':models.conv(),
        'lstm_s':models.lstm(),'fc_s':models.fc(),'regression_s':models.regression(),
        'discriminator':models.Discriminator(),
    }
    opts = {k: optim.Adam(mdls[k].parameters(), lr=LR)
            for k in ['conv','lstm','fc','regression','discriminator']}
    t0 = time.time()
    pretrain(
        rundir=f'source_{temp}',
        source_temp=temp, target_temp=temp,
        source_data_path=OUTPUT_BASE + '/',
        source_train_set=mydata.Pan_all_set,
        source_test_set=mydata.Pan_all_set,
        models=mdls, criterion=nn.MSELoss(reduction='sum'),
        optimizers=opts, batch_size=BATCH_SIZE, epochs=EPOCHS,
        eval_interval=EVAL_INTERVAL, seed=100,
        device_type=DEVICE, ifsave=True, load_model=False,
    )
    elapsed = (time.time()-t0)/60
    run_dirs = glob.glob(os.path.join(WORK_DIR, f'run/source_{temp}*'))
    if run_dirs:
        run_dir = run_dirs[0]
        # 保存模型
        pt = os.path.join(run_dir, 'saved_model/best.pt')
        if os.path.exists(pt):
            shutil.copy(pt, os.path.join(DRIVE_SRC_MODELS, f'pre-{temp}.pt'))
            print(f'[{temp}°C] 模型已保存: pre-{temp}.pt  ({elapsed:.1f} min)')
        # 保存评测指标
        min_err = os.path.join(run_dir, 'errors/min_errors')
        if os.path.exists(min_err):
            shutil.copy(min_err, os.path.join(DRIVE_SRC_MODELS, f'metrics_{temp}.txt'))
            with open(min_err) as f:
                lines = f.readlines()
            print(f'[{temp}°C] 指标: {lines[0].strip()}  {lines[1].strip()}')
        # 保存 loss 曲线图
        for img in glob.glob(os.path.join(run_dir, 'loss_iter/**/*.jpg'), recursive=True):
            shutil.copy(img, os.path.join(DRIVE_SRC_MODELS, f'loss_{temp}_{os.path.basename(img)}'))
    else:
        print(f'[{temp}°C] 未找到 run 目录')

print('\n全部源域模型训练完成')
print('Drive 保存内容：pre-{temp}.pt（模型）+ metrics_{temp}.txt（指标）+ loss 曲线图')

Streaming output truncated to the last 5000 lines.
epoch 1177:loss 0.013051125956209083
epoch 1178:loss 0.01346150143562179
epoch 1179:loss 0.015660477802157402
epoch 1180:loss 0.019930940807649965
epoch 1181:loss 0.015506225025379345
epoch 1182:loss 0.011154616741757644
epoch 1183:loss 0.011564501904343305
epoch 1184:loss 0.00959070526847714
epoch 1185:loss 0.009593236897336809
epoch 1186:loss 0.01084543294028232
epoch 1187:loss 0.010796972158315935
epoch 1188:loss 0.009148700750972094
min loss:0.009148700750972094 saved model
mae = 0.004268334712833166 rmse = 0.005580408618148889 max = 0.02342824637889862
min avg loss:0.004924371665491027 saved model
epoch 1189:loss 0.009722700685654817
epoch 1190:loss 0.01367645093092793
epoch 1191:loss 0.09265722706913948
epoch 1192:loss 0.10892526256410699
epoch 1193:loss 0.13297083973884583
epoch 1194:loss 0.09880496090964268
epoch 1195:loss 0.0974443162742414
epoch 1196:loss 0.06705855852679203
epoch 1197:loss 0.04187092792830969
epoch 1198:loss

## 结果对比：Table II&III（论文 vs 复现）

In [7]:

# ============================================================
# Step 7 源域预训练结果 — 对比论文 Table II（训练自评估）
# 注：本结果在全部 9 个 cycle 上训练+评估（无独立测试集），
#     因此误差偏低；论文为留出测试集后的真实训练误差。
# ------------------------------------------------------------
# 温度     我们 MAE   论文 MAE   我们 RMSE  论文 RMSE
# 25°C     0.162%    0.323%     0.204%     0.404%
# 10°C     0.219%    0.280%     0.300%     0.342%
#  0°C     0.177%    0.188%     0.225%     0.226%
# -10°C    0.297%    0.164%     0.355%     0.208%
# -20°C    0.317%    0.515%     0.410%     0.652%
# ------------------------------------------------------------
# 平均      0.234%    0.294%     0.299%     0.366%
# ============================================================

In [ ]:
# ============================================================
# Table III — Pretraining Estimation Results (CNN+BiLSTM)
# 留一法（LOO）测试误差对比论文
# ------------------------------------------------------------
# Temp      Paper MAE   Ours MAE    Δ MAE    Paper RMSE  Ours RMSE   Δ RMSE
# 25°C        0.760%      1.655%   +0.895%      1.070%     2.694%   +1.624%
# 10°C        1.370%      2.448%   +1.078%      1.810%     3.439%   +1.629%
#  0°C        0.535%      1.326%   +0.791%      0.689%     2.576%   +1.887%
# -10°C       1.110%      2.415%   +1.305%      1.460%     4.108%   +2.648%
# -20°C       1.690%      3.399%   +1.709%      2.180%     5.285%   +3.105%
# ------------------------------------------------------------
# Avg         1.090%      2.249%   +1.159%      1.440%     3.620%   +2.180%
# ============================================================
# 复现结果约为论文的 2.1x (MAE)，2.5x (RMSE)
# 主要差距来自低温（n10, n20）数据量少；0°C 最接近论文（Δ MAE +0.791%）
# ============================================================